<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 2: Working with Text Data

Packages that are being used in this notebook:

In [11]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt")
    file_path = "the-verdict.txt"
    urrllib.request.urlretrieve(url, file_path)


In [12]:
with open("the-verdict.txt", "r", encoding="UTF-8") as f:
    raw_text = f.read()

In [13]:
#raw_text

In [14]:
len(raw_text)

20479

In [15]:
import re

text = "Hello, world. This, is a tests."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'tests.']


In [16]:
result = re.split(r'([,.]\s)', text)
print(result)

['Hello', ', ', 'world', '. ', 'This', ', ', 'is a tests.']


In [17]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ', ', 'world', '. ', 'This', ', ', 'is a tests.']


In [18]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result
#print(preprocessed)

In [19]:
len(preprocessed)

4690

## 2.3 Converting tokens into token IDs

In [20]:
vocab["Jack"] # Token ID for "Jack"

NameError: name 'vocab' is not defined

In [ ]:
int_to_str = {i:s for s, i in vocab.items()}

int_to_str[57]

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

In [ ]:
vocab = {token:integer for integer, token in enumerate(all_words)}
#vocab

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Make Token IDs every token in 'preprocessed'
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """
        Decode the tokenized text back to 'preprocessed'
        """
        
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

In [ ]:
text = """ "It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""

In [ ]:
ids = tokenizer.encode(text)
print(ids)

In [ ]:
tokenizer.decode(ids)

## 2.4 Adding special context tokens

In [ ]:
text = "Hello, do you like tea? is this-- a test?"

tokenizer.encode(text)

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

In [ ]:
len(vocab.items())

In [ ]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?_!"()\']|--|\s)', text)
        
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]

        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        
        # Make Token IDs every token in 'preprocessed'
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """
        Decode the tokenized text back to 'preprocessed'
        """
        
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

In [22]:
tokenizer.encode(text)

NameError: name 'tokenizer' is not defined

In [23]:
text

'Hello, world. Is this-- a test?'

In [24]:
tokenizer.decode(tokenizer.encode(text))

NameError: name 'tokenizer' is not defined

## 2.5 Byte Pair encoding

In [25]:
import tiktoken # Import GPT-2 tokenizer

In [26]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [33]:
tiktoken.__version__

'0.12.0'

In [34]:
tokenizer = tiktoken.get_encoding("gpt2")

In [35]:
tokenizer.encode("Hello, World")

[15496, 11, 2159]

In [36]:
tokenizer.decode(tokenizer.encode("Hello, World"))

'Hello, World'

In [37]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
)

tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114]

## 2.6 Data sampling with a sliding window

In [39]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [42]:
enc_sample = enc_text[50:]

In [43]:
len(enc_sample)

5095

In [47]:
context_size = 4 # Split to make amount managable

# Shift it by one position
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

# x should predict y, the next token
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [48]:
# More visualization
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "----->", desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [51]:
# Even more visualization
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [52]:
import torch

In [54]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [55]:
torch.__version__

'2.10.0+cpu'

In [64]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext>|"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length (see above)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [65]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset object
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [66]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [72]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [73]:
second_batch = next(data_iter)
print(second_batch)

# Moves forward in a stride of 1, with a context length of 4 (max_length)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [74]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings

In [99]:
input_ids = torch.tensor([2,   3,    5,   1])

In [117]:
vocab_size = 6
output_dim = 3

#torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, output_dim)

In [118]:
print(embedding_layer.weight)

Parameter containing:
tensor([[-0.3482,  0.0066,  0.1852],
        [ 1.9515,  1.5087, -0.5176],
        [ 0.5618,  3.0094, -0.2286],
        ...,
        [-1.1224,  0.0202, -0.4359],
        [ 0.5129, -0.8820,  0.6062],
        [ 0.6012,  0.9536, -1.6710]], requires_grad=True)


In [123]:
embedding_layer(torch.tensor([3]))

# The embeddings, is a vector with values that are random at first
# Then trained to be able to predict the right next words

tensor([[-0.9177, -0.0855,  1.0832]], grad_fn=<EmbeddingBackward0>)

In [120]:
embedding_layer(torch.tensor([3]))

tensor([[-0.9177, -0.0855,  1.0832]], grad_fn=<EmbeddingBackward0>)

In [121]:
embedding_layer(input_ids)

tensor([[ 0.5618,  3.0094, -0.2286],
        [-0.9177, -0.0855,  1.0832],
        [ 0.5536,  0.7399, -0.9308],
        [ 1.9515,  1.5087, -0.5176]], grad_fn=<EmbeddingBackward0>)

In [122]:
input_ids

tensor([2, 3, 5, 1])

## 2.8 Encoding word positions

In [124]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [125]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
imputs, targets = next(data_iter)

In [126]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [137]:
token_embeddings = token_embedding_layer(input_ids)
token_embeddings.shape

torch.Size([4, 256])

In [139]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [150]:
# How the token embedding of token ID '40' looks like
# 256 dimensions
# Question: Why does it need to split a token ID that many times?

#token_embeddings[0, 0]

In [146]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [147]:
torch.arange(max_length)

tensor([0, 1, 2, 3])

In [148]:
pos_embedding_layer.weight

Parameter containing:
tensor([[-0.6132, -0.5736,  0.3847,  ..., -2.2183,  0.2349, -1.1365],
        [ 0.4858,  0.1352,  1.2582,  ..., -2.3439,  0.0212,  0.2686],
        [ 0.9462,  0.3237, -0.7339,  ...,  0.4183, -0.8643,  0.0876],
        [ 1.5904,  0.5822, -0.0277,  ...,  1.2015,  0.8532, -1.3093]],
       requires_grad=True)

In [149]:
pos_embedding_layer(torch.arange(max_length))

tensor([[-0.6132, -0.5736,  0.3847,  ..., -2.2183,  0.2349, -1.1365],
        [ 0.4858,  0.1352,  1.2582,  ..., -2.3439,  0.0212,  0.2686],
        [ 0.9462,  0.3237, -0.7339,  ...,  0.4183, -0.8643,  0.0876],
        [ 1.5904,  0.5822, -0.0277,  ...,  1.2015,  0.8532, -1.3093]],
       grad_fn=<EmbeddingBackward0>)

In [152]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


In [154]:
token_embeddings.shape

torch.Size([8, 4, 256])

In [155]:
pos_embeddings.shape

torch.Size([4, 256])

In [156]:
token_embeddings[0] + pos_embeddings

tensor([[-0.7912, -2.5078,  0.2718,  ..., -3.7590,  0.3414, -0.3086],
        [ 1.4521, -1.6010,  1.0753,  ..., -1.6707,  0.0846,  0.6284],
        [ 0.4491,  0.2322, -1.5265,  ...,  1.9612, -3.4767,  0.0378],
        [ 0.9242,  0.5201, -1.1537,  ...,  1.6142,  0.6454, -1.2346]],
       grad_fn=<AddBackward0>)

In [158]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


### Adds additional positional input to each embedding to differentiate identical words with eachother based on where they are.